### Apply CHMM

use annasenv1 to be able to utlise gpu 

In [9]:
import os
os.chdir("/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks") #sets working directory to the repo, so that all imports work correctly
print(os.getcwd())
from glob import glob
import numpy as np
import pickle
from modules import hmm
from osl_dynamics import analysis, inference
from osl_dynamics.data import Data


import tensorflow as tf
# List GPUs
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)  # must happen first

/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks


In [ ]:
# LOAD IN DATA -------------------------------------------------------------
files = sorted(
    f for f in glob("/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/osl/*/lcmv-parc-raw.fif")
    if "ses-003" in f #or "ses-004" in f
)
files = files[:4]

data = Data(files, picks="misc", reject_by_annotation="omit", n_jobs=16)
data = hmm.prepare_data_for_canonical_hmm(data, parcellation="Glasser52")

In [12]:
#CREATE OUTPUT DIRECTORIES -------------------------------------------------
# # Create output directory
hmm_dir = f"/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest"
os.makedirs(hmm_dir, exist_ok=True)
alp_dir = f"{hmm_dir}/alp"
os.makedirs(alp_dir, exist_ok=True)
alp_dir_pkl = f"{hmm_dir}/alp_pkl"
os.makedirs(alp_dir_pkl, exist_ok=True)
stc_dir = f"{hmm_dir}/stc"
os.makedirs(stc_dir, exist_ok=True)
figs_dir =f"{hmm_dir}/figures"
os.makedirs(figs_dir, exist_ok=True)
summary_stats_dir =f"{hmm_dir}/summary_stats"
os.makedirs(summary_stats_dir, exist_ok=True)

# LOAD MODEL -------------------------------------------------------------
model = hmm.load_canonical_hmm(n_states=8, parcellation="Glasser52")
#model.summary()

# ESTIMATE STATE PROBABILITIES --------------------------------------------
alp = model.get_alpha(data)
pickle.dump(alp, open(f"{alp_dir_pkl}/alp.pkl", "wb"))
stc = inference.modes.argmax_time_courses(alp)

#CALC ALPS AND STCS AND SAVE AS NPY FILES ---------------------------------
alp_raw_list = []
stc_list = []

for i, parc_fif in enumerate(files):
    # Convert each alpha time series for each file
    alpha_raw = inference.modes.convert_to_mne_raw(
        alp[i],         # Corresponding alpha for this file , #for single subject just alp not alp[i]
        parc_fif,       # Single file (lcmv-parc-raw.fif), not a list
        n_embeddings=data.n_embeddings,
    )
    parc_filename = parc_fif.split("/")[-1]
    id = parc_fif.split("/")[-2]
    output_filename = f"{alp_dir}/{parc_filename.replace('lcmv-parc-raw', f'{id}_alpha-raw')}"
    alpha_raw.save(output_filename, overwrite=True)
    alp_raw_list.append(alpha_raw)

    # Binarise the state probabilities
    #stc = inference.modes.argmax_time_courses(alp[i])
    stc_raw = inference.modes.convert_to_mne_raw(
        stc[i],         
        parc_fif,       # Single file (lcmv-parc-raw.fif), not a list
        n_embeddings=data.n_embeddings,
    )
    
    output_filename_stc = f"{stc_dir}/{parc_filename.replace('lcmv-parc-raw', f'{id}_stc-raw')}"
    stc_raw.save(output_filename_stc, overwrite=True)
    stc_list.append(alpha_raw)

    print("saved" + str(i))

# CALCULATE MULTITAPER-------------------------------------------------
trimmed_data = data.trim_time_series(sequence_length=model.config.sequence_length, prepared=False)

f, psd, coh, w = analysis.spectral.multitaper_spectra(
    data=trimmed_data,
    alpha=alp,
    sampling_frequency=250,
    frequency_range=[0.5, 45],
    return_weights=True,
)
# Save
np.save(f"{summary_stats_dir}/f.npy", f)
np.save(f"{summary_stats_dir}/psd.npy", psd)
np.save(f"{summary_stats_dir}/coh.npy", coh)
np.save(f"{summary_stats_dir}/w.npy", w)

# CALC POW_MAPS, COH_MAPS AND SUMMARY STATS AND SAVE AS NPYS --------------
# Power maps
pow_maps = analysis.power.variance_from_spectra(f, psd)
np.save(f"{summary_stats_dir}/pow_maps.npy", pow_maps)

# Coherence networks
coh_nets = analysis.connectivity.mean_coherence_from_spectra(f, coh)
np.save(f"{summary_stats_dir}/coh_nets.npy", coh_nets)

# Summary statistics
fo = analysis.post_hoc.fractional_occupancies(stc)
lt = analysis.post_hoc.mean_lifetimes(stc, sampling_frequency=alp_raw_list[0].info["sfreq"])
intv = analysis.post_hoc.mean_intervals(stc, sampling_frequency=alp_raw_list[0].info["sfreq"])
sr = analysis.post_hoc.switching_rates(stc, sampling_frequency=alp_raw_list[0].info["sfreq"])

# Save
np.save(f"{summary_stats_dir}/fo.npy", fo)
np.save(f"{summary_stats_dir}/lt.npy", lt)
np.save(f"{summary_stats_dir}/intv.npy", intv)
np.save(f"{summary_stats_dir}/sr.npy", sr)



I0000 00:00:1775039109.746399  507349 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 27319 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:02:00.0, compute capability: 12.0
I0000 00:00:1775039109.760949  507349 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x58c8218ec4c0


Getting alpha:   0%|          | 0/4 [00:00<?, ?it/s]

2026-04-01 11:25:11.550035: I external/local_xla/xla/service/service.cc:153] XLA service 0x705314003ba0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-01 11:25:11.550052: I external/local_xla/xla/service/service.cc:161]   StreamExecutor device (0): NVIDIA GeForce RTX 5090, Compute Capability 12.0
2026-04-01 11:25:11.597219: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775039111.723249  512193 cuda_dnn.cc:529] Loaded cuDNN version 91900
I0000 00:00:1775039112.346657  512193 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-04-01 11:25:14.997300: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-01 11:25:17.234215: I tensorflow/core/framework/local_rendezvous.cc:4

Writing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest/alp/sub-001_ses-003_task-braille_run-001_alpha-raw.fif
Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest/alp/sub-001_ses-003_task-braille_run-001_alpha-raw.fif
[done]
Writing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest/stc/sub-001_ses-003_task-braille_run-001_stc-raw.fif
Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest/stc/sub-001_ses-003_task-braille_run-001_stc-raw.fif
[done]
saved0
Writing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and_squids/hmm_8state_scripttest/alp/sub-002_ses-003_task-braille_run-001_alpha-raw.fif
Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_old/derivatives_opms_and

Calculating spectra:   0%|          | 0/4 [00:00<?, ?it/s]